# MoodNote-AI — Fine-tune PhoBERT trên Google Colab

Pipeline phân loại cảm xúc tiếng Việt với **PhoBERT + UIT-VSMEC + ViGoEmotions**.

**Trước khi chạy:**
1. Vào `Runtime → Change runtime type → T4 GPU`
2. Vào `Runtime → Manage Secrets` → thêm secret key **`HF_TOKEN`** (token HuggingFace của bạn)
3. Thay `REPO_URL` ở Cell 3 bằng URL GitHub của bạn
4. Chạy `Runtime → Run all`

## Cell 1 — Kiểm tra GPU

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "Không tìm thấy GPU!\n"
        "Vào Runtime → Change runtime type → chọn T4 GPU rồi chạy lại."
    )

gpu_name = torch.cuda.get_device_name(0)
gpu_mem  = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"GPU   : {gpu_name}")
print(f"VRAM  : {gpu_mem:.1f} GB")
print(f"PyTorch: {torch.__version__}")
print("Sẵn sàng train!")

## Cell 2 — Mount Google Drive

Model và checkpoint sẽ được lưu vào Drive để không mất khi session kết thúc.

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

# Tạo thư mục lưu trữ trong Drive
DRIVE_BASE = '/content/drive/MyDrive/MoodNote-AI'
os.makedirs(f'{DRIVE_BASE}/checkpoints', exist_ok=True)
os.makedirs(f'{DRIVE_BASE}/best_model',  exist_ok=True)

print(f"Drive mounted. Thư mục lưu: {DRIVE_BASE}")

## Cell 3 — Clone repo & cài dependencies

> **Thay `REPO_URL`** bằng URL GitHub của bạn, ví dụ:
> `https://github.com/username/MoodNote-AI.git`

In [ ]:
import sys

REPO_URL  = 'https://github.com/ToanHuynh0201/MoodNote-AI.git'  # <-- ĐỔI Ở ĐÂY
REPO_DIR  = '/content/MoodNote-AI'

# Clone repo
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print(f"Repo đã tồn tại tại {REPO_DIR}, bỏ qua clone.")
    !cd {REPO_DIR} && git pull

# Thêm vào Python path
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# Cài dependencies
print("\nCài đặt dependencies...")
!pip install -r {REPO_DIR}/requirements.txt -q

# Cài YAKE cho keyword extraction (FR-13)
print("Cài đặt YAKE...")
!pip install yake -q

print("\nHoàn tất cài đặt!")

## Cell 4 — Cấu hình paths cho Colab

In [ ]:
import os

# --- Paths ---
REPO_DIR       = '/content/MoodNote-AI'
CONFIG_DIR     = f'{REPO_DIR}/configs'
RAW_DIR        = f'{REPO_DIR}/data/raw'
MERGED_DIR     = f'{REPO_DIR}/data/merged'
PROCESSED_DIR  = f'{REPO_DIR}/data/processed'
CHECKPOINT_DIR = f'{DRIVE_BASE}/checkpoints'
BEST_MODEL_DIR = f'{DRIVE_BASE}/best_model'

os.makedirs(RAW_DIR,       exist_ok=True)
os.makedirs(MERGED_DIR,    exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)

# --- Tắt W&B ---
os.environ['WANDB_MODE']    = 'disabled'
os.environ['WANDB_SILENT']  = 'true'

print("Paths đã cấu hình:")
print(f"  Config     : {CONFIG_DIR}")
print(f"  Data raw   : {RAW_DIR}")
print(f"  Merged     : {MERGED_DIR}")
print(f"  Processed  : {PROCESSED_DIR}")
print(f"  Checkpoints: {CHECKPOINT_DIR}")
print(f"  Best model : {BEST_MODEL_DIR}")
print("  W&B        : disabled")

## Cell 5 — Download UIT-VSMEC

In [ ]:
import os
os.chdir(REPO_DIR)

from src.data.download_dataset import download_uit_vsmec

print("=" * 50)
print("Bước 1: Download UIT-VSMEC")
print("=" * 50)
download_uit_vsmec(output_dir=RAW_DIR)

import pandas as pd
for split in ['train', 'validation', 'test']:
    f = f'{RAW_DIR}/{split}.csv'
    df = pd.read_csv(f)
    print(f"  {split:12s}: {len(df):,} mẫu")

## Cell 5.2 — Download ViGoEmotions (gated dataset)

> **Trước khi chạy:** vào `Runtime → Manage Secrets` → thêm secret với key **`HF_TOKEN`** và value là HuggingFace token của bạn (lấy tại [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens)).

In [ ]:
import os
os.chdir(REPO_DIR)

# Đọc HF token từ Colab Secrets (Runtime → Manage Secrets → thêm key "HF_TOKEN")
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
    print("HF token loaded from Colab Secrets.")
except Exception:
    HF_TOKEN = None
    print("Warning: Không lấy được token từ Colab Secrets.")
    print("Nếu download thất bại, thêm secret 'HF_TOKEN' trong Runtime → Manage Secrets.")

from src.data.download_vigoemotions import download_vigoemotions

print("\n" + "=" * 50)
print("Bước 2: Download ViGoEmotions")
print("=" * 50)
download_vigoemotions(output_dir=RAW_DIR, token=HF_TOKEN)

## Cell 5.3 — Merge UIT-VSMEC + ViGoEmotions

Gộp 2 dataset thành một:
- **train / validation**: VSMEC + ViGoEmotions (mapped + deduplicated)
- **test**: VSMEC only (giữ clean benchmark)

In [ ]:
import os
os.chdir(REPO_DIR)

from src.data.merge_datasets import main as merge_main

print("=" * 50)
print("Bước 3: Merge UIT-VSMEC + ViGoEmotions")
print("=" * 50)
merge_main(
    vsmec_dir=RAW_DIR,
    vigoemotions_dir=f'{RAW_DIR}/vigoemotions',
    output_dir=MERGED_DIR,
)

## Cell 5.4 — Preprocess merged dataset (word segmentation)

In [ ]:
import os
os.chdir(REPO_DIR)

from src.data.preprocess import preprocess_dataset

print("=" * 50)
print("Bước 4: Preprocess (word segmentation)")
print("=" * 50)
preprocess_dataset(
    input_dir=MERGED_DIR,
    output_dir=PROCESSED_DIR,
    config_path=f'{CONFIG_DIR}/model_config.yaml'
)

import pandas as pd
for split in ['train', 'validation', 'test']:
    f = f'{PROCESSED_DIR}/{split}.csv'
    df = pd.read_csv(f)
    print(f"  {split:12s}: {len(df):,} mẫu — {f}")

## Cell 5.5 — Data Augmentation (minority classes)

Tăng số lượng mẫu cho Anger, Fear, Surprise bằng Random Deletion và Random Swap.
Không augment class **Other** vì labels mơ hồ.

In [ ]:
import os
os.chdir(REPO_DIR)

import pandas as pd
from src.data.augment import augment_dataset

# Xem phân bổ class sau merge để điều chỉnh target_counts nếu cần
train_df = pd.read_csv(f'{PROCESSED_DIR}/train.csv')
print("Phân bổ class sau merge + preprocess:")
print(train_df['label'].value_counts().sort_index())
median_count = int(train_df['label'].value_counts().median())
print(f"Median count: {median_count}")

# Augment các class có count < 60% median
from collections import Counter
dist = Counter(train_df['label'].tolist())
target_counts = {
    idx: int(median_count * 0.8)
    for idx, cnt in dist.items()
    if cnt < median_count * 0.6
}
print(f"\nAugment target_counts: {target_counts}")

if target_counts:
    augment_dataset(
        input_csv=f"{PROCESSED_DIR}/train.csv",
        output_csv=f"{PROCESSED_DIR}/train_augmented.csv",
        target_counts=target_counts,
        techniques=["deletion", "swap"]
    )
else:
    print("Dataset đã cân bằng, không cần augmentation. Copy train.csv → train_augmented.csv")
    import shutil
    shutil.copy(f"{PROCESSED_DIR}/train.csv", f"{PROCESSED_DIR}/train_augmented.csv")

aug_df = pd.read_csv(f'{PROCESSED_DIR}/train_augmented.csv')
print(f"\nSau augmentation: {len(aug_df):,} mẫu")

## Cell 6 — Train model

Quá trình train 15 epoch, khoảng **60-90 phút** trên T4 GPU.

In [ ]:
import os, torch
import pandas as pd
os.chdir(REPO_DIR)

from src.data.dataset import EmotionDataset
from src.models.phobert_classifier import PhoBERTEmotionClassifier
from src.models.model_utils import save_model, get_device, print_model_summary
from src.training.trainer import train_model
from src.utils.config import load_all_configs
from src.utils.logger import setup_logger
from src.utils.metrics import compute_metrics, print_metrics, plot_confusion_matrix
import numpy as np
from pathlib import Path

logger = setup_logger(name='colab_training')

# Load configs
configs       = load_all_configs(CONFIG_DIR)
model_config  = configs['model']
training_config = configs['training']

logger.info(f"Model         : {model_config['model']['name']}")
logger.info(f"Epochs        : {training_config['training']['num_epochs']}")
logger.info(f"Batch size    : {training_config['training']['batch_size']}")
logger.info(f"LR            : {training_config['training']['learning_rate']}")
logger.info(f"Focal gamma   : {model_config['model'].get('focal_gamma', 2.0)}")

device = get_device()

# Datasets
model_name = model_config['model']['name']
max_len    = model_config['model']['max_seq_length']

train_dataset = EmotionDataset(f'{PROCESSED_DIR}/train_augmented.csv',      tokenizer_name=model_name, max_length=max_len)
val_dataset   = EmotionDataset(f'{PROCESSED_DIR}/validation.csv', tokenizer_name=model_name, max_length=max_len)
test_dataset  = EmotionDataset(f'{PROCESSED_DIR}/test.csv',       tokenizer_name=model_name, max_length=max_len)

logger.info(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_dataset)}")

# Class weights (xử lý dataset mất cân bằng)
train_labels = pd.read_csv(f'{PROCESSED_DIR}/train_augmented.csv')['label'].tolist()
class_counts = np.bincount(train_labels, minlength=7).astype(float)
class_weights = torch.tensor(1.0 / class_counts, dtype=torch.float32)
class_weights = class_weights / class_weights.sum() * 7
print("Class weights:")
for i, w in enumerate(class_weights):
    print(f"  {model_config['emotion_labels'][i]:<12}: {w:.3f}")

# Model
model = PhoBERTEmotionClassifier(
    model_name=model_name,
    num_labels=model_config['model']['num_labels'],
    dropout=model_config['model']['dropout'],
    class_weights=class_weights,
    label_smoothing=model_config['model'].get('label_smoothing', 0.0),
    focal_gamma=model_config['model'].get('focal_gamma', 2.0)
)
model.to(device)
print_model_summary(model)

# Train
trainer = train_model(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    training_config=training_config,
    output_dir=CHECKPOINT_DIR,
    use_wandb=False
)

# Evaluate on test set
logger.info("Evaluating on test set...")
predictions   = trainer.predict(test_dataset)
test_preds    = predictions.predictions
test_labels   = predictions.label_ids

detailed = compute_metrics(test_preds, test_labels)
print_metrics(detailed, model_config['emotion_labels'])

# Confusion matrix
cm_path = Path(CHECKPOINT_DIR) / 'confusion_matrix.png'
plot_confusion_matrix(test_preds, test_labels,
                      emotion_labels=model_config['emotion_labels'],
                      save_path=cm_path)

# Save best model vào Drive
save_model(
    model=trainer.model,
    tokenizer=train_dataset.tokenizer,
    save_dir=BEST_MODEL_DIR,
    config={
        'model_config': model_config,
        'training_config': training_config,
        'test_results': {
            'accuracy':    detailed['accuracy'],
            'f1_macro':    detailed['f1_macro'],
            'f1_weighted': detailed['f1_weighted']
        }
    }
)

print("\n" + "=" * 50)
print("TRAINING HOÀN TẤT")
print("=" * 50)
print(f"Accuracy   : {detailed['accuracy']:.4f}")
print(f"F1-Macro   : {detailed['f1_macro']:.4f}")
print(f"F1-Weighted: {detailed['f1_weighted']:.4f}")
print(f"Model đã lưu tại: {BEST_MODEL_DIR}")

## Cell 7 — Kiểm tra model & test predict

In [ ]:
import os
os.chdir(REPO_DIR)

from src.models.model_utils import load_model
from src.inference.predictor import EmotionPredictor

# Kiểm tra file đã lưu trong Drive
print("Files trong best_model:")
for f in sorted(os.listdir(BEST_MODEL_DIR)):
    size = os.path.getsize(f'{BEST_MODEL_DIR}/{f}') / 1024**2
    print(f"  {f:<30} {size:.1f} MB")

# Test predict
print("\nTest predict:")
predictor = EmotionPredictor(model_path=BEST_MODEL_DIR)

test_sentences = [
    "Hôm nay tôi rất vui vì được nghỉ học!",
    "Tôi buồn quá, không biết phải làm sao.",
    "Thật tức giận khi bị đối xử bất công.",
    "Trời ơi, tin này làm tôi bất ngờ quá!",
]

for sentence in test_sentences:
    result = predictor.predict(sentence)
    print(f"  Text      : {sentence}")
    print(f"  Emotion   : {result['emotion']}")
    print(f"  Confidence: {result['confidence']:.2%}")
    print()